# 📅 DAY 1 — Using Pretrained Transformers (Industry Usage)

By the end of this notebook, you will:
- Understand why industry prefers pretrained transformers
- Use Hugging Face pipelines for NLP tasks
- Run inference on CPU and GPU
- Measure performance and scalability


## Why Industry Uses Pretrained Transformers

### Traditional ML Approach
1. Data collection
2. Feature engineering
3. Model training
4. Hyperparameter tuning
5. Deployment

Time-consuming  
- Expensive  
- High risk

### Transformer-Based Industry Approach
1. Choose pretrained model
2. Load using `pipeline()`
3. Run inference
4. Deploy instantly


## Hugging Face Ecosystem

Hugging Face is the **GitHub of AI models**.

### What it provides:
- Pretrained transformer models
- Ready-to-use pipelines
- Tokenizers
- Model Hub

### Industry Insight
Companies do not train models from scratch.
They **reuse and deploy**.


In [ ]:
!pip install transformers torch

## Transformers vs Traditional ML Models

| Feature | Traditional ML | Transformers |
|------|--------------|-------------|
| Feature Engineering | Manual | Automatic |
| Context Understanding | Weak | Strong |
| Accuracy | Medium | Very High |
| Deployment | Slow | Fast |

Transformers understand **context**, not keywords.


## Hugging Face Pipelines

A pipeline combines:
- Tokenization
- Model inference
- Output processing

All in **one line of code**.

This is why pipelines are heavily used in production systems.


In [ ]:
from transformers import pipeline
import torch
import time

## GPU Availability Check

Before running inference, industry systems always check GPU availability.


In [ ]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))


CUDA Available: False


## Device Selection

- `device = -1` → CPU
- `device = 0` → GPU

This allows safe fallback in production systems.


In [ ]:
device = 0
if torch.cuda.is_available() :
  device = 0
else:
  device = -1
print("Using device:", device)


Using device: 0


## Sentiment Analysis — Customer Reviews

Used by:
- E-commerce platforms
- App reviews
- Customer feedback systems


In [ ]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    device=device
)

reviews = [
    "The product quality is amazing!",
    "Worst customer service ever.",
    "Delivery was late but product is good."
]

results = sentiment_pipeline(reviews)

for review, result in zip(reviews, results):
    print(review)
    print(result)
    print("-" * 50)


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0


The product quality is amazing!
{'label': 'POSITIVE', 'score': 0.9998865127563477}
--------------------------------------------------
Worst customer service ever.
{'label': 'NEGATIVE', 'score': 0.999788224697113}
--------------------------------------------------
Delivery was late but product is good.
{'label': 'POSITIVE', 'score': 0.9994235038757324}
--------------------------------------------------


## Toxic Comment Detection — Social Media Monitoring

Used for:
- Hate speech detection
- Comment moderation
- Gaming platforms


In [ ]:
toxic_pipeline = pipeline(
    "text-classification",
    model="unitary/toxic-bert",
    device=device
)

comments = [
    "You are so stupid!",
    "I really appreciate your help",
    "This is the worst thing ever"
]

results = toxic_pipeline(comments)

for comment, result in zip(comments, results):
    print(comment)
    print(result)
    print("-" * 50)


config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


You are so stupid!
{'label': 'toxic', 'score': 0.9881107807159424}
--------------------------------------------------
I really appreciate your help
{'label': 'toxic', 'score': 0.000573008437640965}
--------------------------------------------------
This is the worst thing ever
{'label': 'toxic', 'score': 0.47822117805480957}
--------------------------------------------------


## Text Summarization — Emails & Reports

Used in:
- Corporate emails
- Legal documents
- Executive summaries


In [ ]:
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=device
)

text = """
Artificial Intelligence is transforming businesses across industries.
Companies are adopting AI to automate workflows, improve customer experience,
and reduce operational costs. However, successful AI adoption requires
strategic planning, skilled workforce, and ethical considerations.
"""

summary = summarizer(
    text,
    max_length=60,
    min_length=30,
    do_sample=False
)

print(summary[0]["summary_text"])


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
Your max_length is set to 60, but your input_length is only 52. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=26)


Companies are adopting AI to automate workflows, improve customer experience, and reduce operational costs. Successful AI adoption requires strategic planning, skilled workforce, and ethical considerations.


## Batch Inference

GPU inference is most efficient when processing **multiple inputs together**.
This increases throughput and reduces cost.


In [ ]:
batch_reviews = [
    "Excellent product!",
    "Very poor quality.",
    "Worth the price.",
    "Terrible experience.",
    "Loved it!"
]

results = sentiment_pipeline(batch_reviews)

for text, result in zip(batch_reviews, results):
    print(text, "->", result)


Excellent product! -> {'label': 'POSITIVE', 'score': 0.9998724460601807}
Very poor quality. -> {'label': 'NEGATIVE', 'score': 0.9998114705085754}
Worth the price. -> {'label': 'POSITIVE', 'score': 0.9998661279678345}
Terrible experience. -> {'label': 'NEGATIVE', 'score': 0.9996143579483032}
Loved it! -> {'label': 'POSITIVE', 'score': 0.9998828172683716}


## Measuring Inference Performance

Latency is critical in production systems.
We compare CPU vs GPU inference time.


In [ ]:
text = "I love using transformers in production systems."

# CPU
cpu_pipeline = pipeline("sentiment-analysis", device=-1)
start = time.time()
cpu_pipeline(text)
cpu_time = time.time() - start

# GPU
gpu_pipeline = pipeline("sentiment-analysis", device=device)
start = time.time()
gpu_pipeline(text)
gpu_time = time.time() - start

print("CPU Time:", cpu_time)
print("GPU Time:", gpu_time)


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0


CPU Time: 0.17494916915893555
GPU Time: 0.005148887634277344


## Industry Use Cases

| Business Problem | Transformer Task |
|----------------|----------------|
| Product Reviews | Sentiment Analysis |
| Social Media | Toxic Detection |
| Emails | Summarization |
| Customer Support | Text Classification |


## ✅ Hands-On Outcomes

By completing this notebook, you can:
- Use pretrained transformers
- Run inference on GPU
- Perform batch inference
- Measure latency
- Understand industry deployment patterns


## Assignment

1. Run sentiment analysis on 10 real reviews
2. Compare CPU vs GPU inference
3. Test batch size of 5 and 10
4. Write a short conclusion:
   **When is GPU inference worth it?**


In [ ]:
from huggingface_hub import InferenceClient

# Initialize client with your API token
client = InferenceClient(api_key="hf_key")

# Generate the image
# You can specify models like "black-forest-labs/FLUX.1-dev" or "stabilityai/stable-diffusion-xl-base-1.0"
image = client.text_to_image(
    "A futuristic cityscape at sunset with flying cars",
    model="black-forest-labs/FLUX.1-dev"
)

# Save the PIL.Image object
image.save("generated_image.png")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
text_generation = pipeline(
    "text-generation",
    model="gpt2",
    device=device)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
text_generation("Information about transformers ai")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'Information about transformers ai:\n\n#include <iostream> #include <string> #include <thread> #include <vector> int main ( int argc, char * argv []) { char * c ; printf ( " %s: %s\n\n", argv [ 1 ]); }\n\nThis command will run the transformers and return a string representing the array of data in the function.\n\nThe main process of this program is to initialize the transformers and find the value of their parameters. The parameters are:\n\n* The transformers used to convert the data, which is the first argument of the function.\n\n* The transformers used to store the data.\n\n* The transformers used to store the data of the function that will be evaluated after this transformation.\n\n* The transformers used to store the data that will be evaluated if the function returns true.\n\n* The transformers used for decoding the data of the function.\n\n* The transformers used for decoding the data of the function that will be evaluated if the function returns false.\n\n* 

In [ ]:
text

[[{'generated_text': "Information about transformers ai-Miner/MinerAi_MinerAi.cpp:55: Warning: undefined type 'int' in transformers.h:4: Warning: undefined type 'int' in transformers.h:8: Warning: undefined type 'int' in transformers.h:14: Warning: undefined type 'int' in transformers.h:17: Warning: undefined type 'int' in transformers.h:23: Warning: undefined type 'int' in transformers.h:29: Warning: undefined type 'int' in transformers.h:37: Warning: undefined type 'int' in transformers.h:43: Warning: undefined type 'int' in transformers.h:45: Warning: undefined type 'int' in transformers.h:49: Warning: undefined type 'int' in transformers.h:54: Warning: undefined type 'int' in transformers.h:59: Warning: undefined type 'int' in transformers.h:61: Warning: undefined type 'int' in transformers.h:63: Warning: undefined type 'int' in transformers.h:67: Warning: undefined type 'int' in transformers.h:"}]]